# News & Macro Agent Prototype

This notebook runs the News & Macro Agent as an event-driven second-order investment idea generator. You can paste one news article directly, or fetch recent context for a ticker when optional packages and API keys are configured.

In [ ]:
# Optional live-data setup, if needed:
# !pip install yfinance requests openai google-genai

from news_macro_agent import analyze_news_text, collect_context, display_report, generate_report, save_report

TICKER = "MSFT"
COMPANY = "Microsoft"
EVENT_DATE = "2026-04-26"
LOOKBACK_DAYS = 14
NEWS_MODE = "latest"  # use "event_window" for historical case studies

# If using OpenAI in Colab:
# import os
# os.environ["OPENAI_API_KEY"] = "YOUR_KEY"

# If using Gemini API in Colab:
# import os
# os.environ["GEMINI_API_KEY"] = "YOUR_KEY"

# If using Vertex AI in Colab:
# from google.colab import auth
# auth.authenticate_user()

In [ ]:
NEWS_TEXT = """
Microsoft shares move as investors parse earnings outlook and AI spending plans.
Investors focused on guidance quality, margin impact, and whether demand trends justify higher spending.
"""

report = analyze_news_text(
    NEWS_TEXT,
    ticker=TICKER,
    company=COMPANY,
    event_date=EVENT_DATE,
    provider="auto",  # auto: OpenAI, Gemini API, Vertex if project is passed, then fallback
    # provider="vertex",
    # vertex_project="csee4121-488316",
    # vertex_location="us-central1",
)
display_report(report)

## Optional: fetch ticker context

Use this path when `yfinance` or `NEWSAPI_KEY` is configured.

In [ ]:
context = collect_context(
    ticker=TICKER,
    company=COMPANY,
    event_date=EVENT_DATE,
    lookback_days=LOOKBACK_DAYS,
    max_news=10,
    news_mode=NEWS_MODE,
)

context

In [ ]:
fetched_report = generate_report(context)
display_report(fetched_report)

In [ ]:
save_report(report, "news_macro_report.json")
report

## Next integration step

Later, the CIO agent can consume `report` directly because it already contains a structured summary, core insight, investment opportunities, risk notes, citations, and raw context.

## Evaluation: Last Earnings Window for Magnificent Seven

This section evaluates price action around the most recent completed earnings event for the Magnificent Seven. It is a first-pass event-window study: the goal is to understand realized stock movement around earnings before connecting those outcomes to agent-generated reports.

In [ ]:
import warnings
from datetime import timedelta

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

MAG7 = {
    "AAPL": "Apple",
    "MSFT": "Microsoft",
    "AMZN": "Amazon",
    "GOOGL": "Alphabet",
    "META": "Meta Platforms",
    "NVDA": "Nvidia",
    "TSLA": "Tesla",
}

PRE_TRADING_DAYS = 7
POST_TRADING_DAYS = 7
BENCHMARKS = ["SPY", "QQQ"]
TODAY = pd.Timestamp.today(tz="America/New_York").normalize().tz_localize(None)

print(f"Evaluation date: {TODAY.date()}")

In [ ]:
def _to_naive_timestamp(value):
    ts = pd.Timestamp(value)
    if ts.tzinfo is not None:
        ts = ts.tz_convert(None)
    return ts.normalize()


def get_last_completed_earnings_date(ticker, today=TODAY, limit=16):
    """Return the latest earnings date that is not in the future."""
    try:
        earnings = yf.Ticker(ticker).get_earnings_dates(limit=limit)
    except Exception as exc:
        return None, f"get_earnings_dates failed: {exc}"

    if earnings is None or len(earnings) == 0:
        return None, "No earnings dates returned by yfinance."

    dates = []
    for value in earnings.index:
        try:
            ts = _to_naive_timestamp(value)
        except Exception:
            continue
        if ts <= today:
            dates.append(ts)

    if not dates:
        return None, "No completed earnings date found."
    return max(dates), None


def _extract_close(downloaded):
    if downloaded is None or downloaded.empty:
        return pd.Series(dtype=float)
    if isinstance(downloaded.columns, pd.MultiIndex):
        if "Close" in downloaded.columns.get_level_values(0):
            close = downloaded["Close"]
        else:
            close = downloaded.xs("Close", axis=1, level=-1, drop_level=False)
        if isinstance(close, pd.DataFrame):
            close = close.iloc[:, 0]
    else:
        close = downloaded.get("Close", pd.Series(dtype=float))
    close = close.dropna()
    close.index = pd.to_datetime(close.index).tz_localize(None).normalize()
    return close


def download_close_series(symbol, event_date, pre_days=PRE_TRADING_DAYS, post_days=POST_TRADING_DAYS):
    """Download enough trading history around an earnings event."""
    start = (event_date - pd.Timedelta(days=max(30, pre_days * 4))).date().isoformat()
    end = (event_date + pd.Timedelta(days=max(30, post_days * 4))).date().isoformat()
    data = yf.download(symbol, start=start, end=end, auto_adjust=True, progress=False)
    return _extract_close(data)


def event_window_from_close(close, event_date, pre_days=PRE_TRADING_DAYS, post_days=POST_TRADING_DAYS):
    """Return a normalized price path indexed by trading-day distance from the pre-earnings anchor."""
    if close.empty:
        return pd.DataFrame(), "No close prices available."

    dates = pd.Series(close.index, index=range(len(close)))
    before = dates[dates < event_date]
    if before.empty:
        return pd.DataFrame(), "No trading day before earnings date."

    anchor_pos = before.index.max()
    start_pos = max(0, anchor_pos - pre_days)
    end_pos = min(len(close) - 1, anchor_pos + post_days)
    window = close.iloc[start_pos : end_pos + 1].copy()
    rel_days = list(range(start_pos - anchor_pos, end_pos - anchor_pos + 1))
    anchor_price = float(close.iloc[anchor_pos])

    result = pd.DataFrame(
        {
            "date": window.index,
            "relative_trading_day": rel_days,
            "close": window.values,
            "normalized_price": window.values / anchor_price,
        }
    )
    return result, None


def value_at_or_after(path, relative_day):
    subset = path[path["relative_trading_day"] >= relative_day]
    if subset.empty:
        return np.nan
    return float(subset.iloc[0]["normalized_price"])


def value_at_or_before(path, relative_day):
    subset = path[path["relative_trading_day"] <= relative_day]
    if subset.empty:
        return np.nan
    return float(subset.iloc[-1]["normalized_price"])


def compute_event_returns(path):
    """Returns are measured from the pre-earnings anchor day, relative day 0."""
    if path.empty:
        return {}
    pre_start = value_at_or_after(path, -PRE_TRADING_DAYS)
    anchor = value_at_or_before(path, 0)
    d1 = value_at_or_after(path, 1)
    d3 = value_at_or_after(path, 3)
    d7 = value_at_or_after(path, POST_TRADING_DAYS)
    return {
        "pre_7d_return": anchor / pre_start - 1 if pre_start and anchor else np.nan,
        "post_1d_return": d1 / anchor - 1 if d1 and anchor else np.nan,
        "post_3d_return": d3 / anchor - 1 if d3 and anchor else np.nan,
        "post_7d_return": d7 / anchor - 1 if d7 and anchor else np.nan,
        "full_window_return": d7 / pre_start - 1 if d7 and pre_start else np.nan,
    }

In [ ]:
earnings_rows = []
for ticker, company in MAG7.items():
    earnings_date, note = get_last_completed_earnings_date(ticker)
    earnings_rows.append(
        {
            "ticker": ticker,
            "company": company,
            "earnings_date": earnings_date.date().isoformat() if earnings_date is not None else None,
            "note": note,
        }
    )

earnings_df = pd.DataFrame(earnings_rows)
earnings_df

In [ ]:
price_paths = []
summary_rows = []

for row in earnings_df.itertuples(index=False):
    if pd.isna(row.earnings_date) or row.earnings_date is None:
        summary_rows.append({"ticker": row.ticker, "company": row.company, "error": row.note})
        continue

    event_date = pd.Timestamp(row.earnings_date)
    ticker_path, ticker_error = event_window_from_close(download_close_series(row.ticker, event_date), event_date)
    if ticker_error:
        summary_rows.append({"ticker": row.ticker, "company": row.company, "earnings_date": row.earnings_date, "error": ticker_error})
        continue

    ticker_path["ticker"] = row.ticker
    ticker_path["company"] = row.company
    ticker_path["symbol_type"] = "stock"
    price_paths.append(ticker_path)

    metrics = compute_event_returns(ticker_path)
    summary = {
        "ticker": row.ticker,
        "company": row.company,
        "earnings_date": row.earnings_date,
        **metrics,
    }

    for benchmark in BENCHMARKS:
        benchmark_path, benchmark_error = event_window_from_close(download_close_series(benchmark, event_date), event_date)
        if benchmark_error:
            summary[f"{benchmark.lower()}_post_7d_return"] = np.nan
            summary[f"abnormal_vs_{benchmark.lower()}"] = np.nan
            continue
        benchmark_metrics = compute_event_returns(benchmark_path)
        summary[f"{benchmark.lower()}_post_7d_return"] = benchmark_metrics.get("post_7d_return", np.nan)
        summary[f"abnormal_vs_{benchmark.lower()}"] = metrics.get("post_7d_return", np.nan) - benchmark_metrics.get("post_7d_return", np.nan)

    summary["realized_direction_vs_qqq"] = "up" if summary.get("abnormal_vs_qqq", np.nan) > 0 else "down"
    summary_rows.append(summary)

summary_df = pd.DataFrame(summary_rows)
return_cols = [col for col in summary_df.columns if "return" in col or "abnormal" in col]
summary_display = summary_df.copy()
for col in return_cols:
    summary_display[col] = summary_display[col].map(lambda x: f"{x:.2%}" if pd.notna(x) else "")
summary_display

In [ ]:
if price_paths:
    paths_df = pd.concat(price_paths, ignore_index=True)

    plt.figure(figsize=(11, 6))
    for ticker, group in paths_df.groupby("ticker"):
        plt.plot(group["relative_trading_day"], group["normalized_price"], marker="o", linewidth=1.8, label=ticker)
    plt.axvline(0, color="black", linestyle="--", linewidth=1, alpha=0.7)
    plt.axhline(1.0, color="gray", linestyle=":", linewidth=1)
    plt.title("Magnificent Seven Normalized Price Path Around Last Earnings")
    plt.xlabel("Trading days relative to pre-earnings anchor day")
    plt.ylabel("Normalized close price, anchor = 1.0")
    plt.legend(ncol=4)
    plt.grid(alpha=0.25)
    plt.show()

    plot_df = summary_df.dropna(subset=["abnormal_vs_qqq"]).sort_values("abnormal_vs_qqq")
    plt.figure(figsize=(10, 5))
    colors = ["#c44e52" if value < 0 else "#4c72b0" for value in plot_df["abnormal_vs_qqq"]]
    plt.bar(plot_df["ticker"], plot_df["abnormal_vs_qqq"], color=colors)
    plt.axhline(0, color="black", linewidth=1)
    plt.title("Post-Earnings 7-Trading-Day Abnormal Return vs QQQ")
    plt.ylabel("Abnormal return")
    plt.gca().yaxis.set_major_formatter(lambda x, _: f"{x:.0%}")
    plt.grid(axis="y", alpha=0.25)
    plt.show()
else:
    print("No price paths available. Check yfinance access or manually provide earnings dates.")

In [ ]:
agent_eval_hook = summary_df[[
    "ticker",
    "company",
    "earnings_date",
    "post_7d_return",
    "abnormal_vs_qqq",
    "realized_direction_vs_qqq",
]].copy()

agent_eval_hook["news_context_ready"] = agent_eval_hook["earnings_date"].notna()
agent_eval_hook["report_ready"] = False
agent_eval_hook["agent_stance"] = None
agent_eval_hook["direction_match"] = None

agent_eval_hook